# Topic Classification with TF-IDF

This notebook implements the topic-analysis part of the Text Mining project. The task is to assign one topic label to each sentence in the provided test set: `book`, `movie`, or `restaurant`.

The project requirement says topic analysis should use TF-IDF and should be performed at sentence level. Following the conventional machine-learning part of `Lab6-assignment-topic-classification.ipynb`, this notebook represents text with TF-IDF features and evaluates predictions with precision, recall, and F1-score.

## Important Data Note

The provided `training_set/Reviews.csv` file contains 5,000 review texts but no topic label column. It also appears to consist mainly of book reviews. Because there is no balanced labelled training set for all three topics, this notebook uses a transparent weak-supervision setup:

- inspect `Reviews.csv` as available training/source data;
- build short topic profile documents for `book`, `movie`, and `restaurant`;
- fit a TF-IDF representation on those topic profiles;
- classify each test sentence by comparing its TF-IDF vector to the topic vectors;
- include a TF-IDF + LinearSVC baseline trained on the same topic profiles, matching the Lab 6 classification style.

This is intentionally simple and interpretable, which is useful for the poster analysis. It also means the model depends heavily on explicit topic words such as *movie*, *book*, *restaurant*, *food*, and *diner*.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

TRAIN_REVIEWS_PATH = Path("training_set/Reviews.csv")
TEST_PATH = Path("test_sets/Sentiment-topic-test.tsv")
OUTPUT_PATH = Path("outputs/topic_tfidf_predictions.csv")
SUMMARY_PATH = Path("outputs/topic_tfidf_summary.csv")

## Load Data and Show Statistics

In [2]:
reviews_df = pd.read_csv(TRAIN_REVIEWS_PATH, encoding="latin1")
test_df = pd.read_csv(TEST_PATH, sep="\t")

display(test_df)
print("Training/source reviews shape:", reviews_df.shape)
print("Test shape:", test_df.shape)
print("Test topic distribution:")
print(test_df["topic"].value_counts())

,sentence id,text,sentiment,topic
0,0,It took eight years for Warner Brothers to rec...,negative,movie
1,1,All the New York University students love this...,positive,restaurant
2,2,This Italian place is really trendy but they h...,negative,restaurant
3,3,"In conclusion, my review of this book would be...",positive,book
4,4,The story of this movie is focused on Carl Bra...,neutral,movie
5,5,Chris O'Donnell stated that while filming for ...,neutral,movie
6,6,My husband and I moved to Amsterdam 6 years ag...,positive,restaurant
7,7,Dame Maggie Smith performed her role excellent...,positive,movie
8,8,The new movie by Mr. Kruno was shot in New Yor...,neutral,movie
9,9,"I always have loved English novels, but I just...",negative,book


Training/source reviews shape: (5000, 1)
Test shape: (10, 4)
Test topic distribution:
topic
movie         5
restaurant    3
book          2
Name: count, dtype: int64


In [3]:
keyword_counts = {}
for keyword in ["book", "novel", "movie", "film", "restaurant", "food", "diner"]:
    keyword_counts[keyword] = int(reviews_df["ReviewContent"].str.contains(keyword, case=False, na=False).sum())

print("Review length statistics, measured in characters:")
display(reviews_df["ReviewContent"].str.len().describe())

print("Keyword counts in Reviews.csv:")
display(pd.Series(keyword_counts, name="review_count").to_frame())

Review length statistics, measured in characters:


count    5000.00000
mean      449.81220
std       367.86781
min         2.00000
25%       251.00000
50%       369.00000
75%       537.00000
max      5520.00000
Name: ReviewContent, dtype: float64

Keyword counts in Reviews.csv:


,review_count
book,3585
novel,648
movie,778
film,71
restaurant,0
food,6
diner,0


The source review file is useful for understanding the available data, but it is not a supervised topic training set because it has no topic labels. The keyword counts show that it is much more book-focused than restaurant-focused, so using it directly as a three-way training set would be misleading.

## TF-IDF Topic Profiles

Each topic is represented by a short profile document containing typical words for that domain. `TfidfVectorizer` converts the profile documents and test sentences to sparse TF-IDF vectors. The predicted topic is the profile with the highest cosine similarity to the sentence vector.

In [4]:
topic_profiles = {
    "book": "book novel novels author writer written reading read pages chapter chapters plot story character characters audiobook narrator literature Jane Austen ending prose mystery bestseller review",
    "movie": "movie movies film films cinema actor actress actors actresses role played filming shot scene scenes story director commercial trailer Navy performed screen character characters",
    "restaurant": "restaurant diner place food eat eating menu dish dishes atmosphere trendy Italian favorite service table students Soho Amsterdam lunch dinner meal cuisine waiter chef",
}

topic_labels = list(topic_profiles.keys())

tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    sublinear_tf=True,
)

profile_matrix = tfidf.fit_transform(topic_profiles.values())
test_matrix = tfidf.transform(test_df["text"])
similarities = cosine_similarity(test_matrix, profile_matrix)

test_df["tfidf_cosine_topic"] = [topic_labels[index] for index in similarities.argmax(axis=1)]
test_df["tfidf_cosine_score"] = similarities.max(axis=1)

display(test_df[["sentence id", "text", "topic", "tfidf_cosine_topic", "tfidf_cosine_score"]])

,sentence id,text,topic,tfidf_cosine_topic,tfidf_cosine_score
0,0,It took eight years for Warner Brothers to rec...,movie,movie,0.148554
1,1,All the New York University students love this...,restaurant,restaurant,0.291730
2,2,This Italian place is really trendy but they h...,restaurant,restaurant,0.326164
3,3,"In conclusion, my review of this book would be...",book,book,0.325081
4,4,The story of this movie is focused on Carl Bra...,movie,movie,0.281015
5,5,Chris O'Donnell stated that while filming for ...,movie,movie,0.257304
6,6,My husband and I moved to Amsterdam 6 years ag...,restaurant,restaurant,0.291730
7,7,Dame Maggie Smith performed her role excellent...,movie,movie,0.257304
8,8,The new movie by Mr. Kruno was shot in New Yor...,movie,movie,0.202484
9,9,"I always have loved English novels, but I just...",book,book,0.145381


## Lab 6-Style Baseline: TF-IDF + LinearSVC

The Lab 6 assignment uses TF-IDF with a conventional classifier such as SVM. Because this project does not include a labelled topic training set, this baseline trains LinearSVC on the same topic profile documents. It is included as a comparison point, not as a fully supervised model.

In [5]:
svm_model = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2), sublinear_tf=True)),
    ("classifier", LinearSVC()),
])

svm_model.fit(list(topic_profiles.values()), topic_labels)
test_df["tfidf_svm_topic"] = svm_model.predict(test_df["text"])

display(test_df[["sentence id", "topic", "tfidf_cosine_topic", "tfidf_svm_topic", "text"]])

,sentence id,topic,tfidf_cosine_topic,tfidf_svm_topic,text
0,0,movie,movie,movie,It took eight years for Warner Brothers to rec...
1,1,restaurant,restaurant,restaurant,All the New York University students love this...
2,2,restaurant,restaurant,restaurant,This Italian place is really trendy but they h...
3,3,book,book,book,"In conclusion, my review of this book would be..."
4,4,movie,movie,movie,The story of this movie is focused on Carl Bra...
5,5,movie,movie,movie,Chris O'Donnell stated that while filming for ...
6,6,restaurant,restaurant,restaurant,My husband and I moved to Amsterdam 6 years ag...
7,7,movie,movie,movie,Dame Maggie Smith performed her role excellent...
8,8,movie,movie,movie,The new movie by Mr. Kruno was shot in New Yor...
9,9,book,book,book,"I always have loved English novels, but I just..."


## Quantitative Evaluation

In [6]:
def summarize_model(model_name, y_true, y_pred):
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {
        "model": model_name,
        "accuracy": report["accuracy"],
        "macro_precision": report["macro avg"]["precision"],
        "macro_recall": report["macro avg"]["recall"],
        "macro_f1": report["macro avg"]["f1-score"],
        "weighted_precision": report["weighted avg"]["precision"],
        "weighted_recall": report["weighted avg"]["recall"],
        "weighted_f1": report["weighted avg"]["f1-score"],
    }


for model_name, prediction_column in [
    ("TF-IDF cosine profiles", "tfidf_cosine_topic"),
    ("TF-IDF + LinearSVC profiles", "tfidf_svm_topic"),
]:
    print(model_name)
    print(classification_report(test_df["topic"], test_df[prediction_column], labels=topic_labels, zero_division=0))

summary_df = pd.DataFrame([
    summarize_model("TF-IDF cosine profiles", test_df["topic"], test_df["tfidf_cosine_topic"]),
    summarize_model("TF-IDF + LinearSVC profiles", test_df["topic"], test_df["tfidf_svm_topic"]),
])
display(summary_df)

TF-IDF cosine profiles
              precision    recall  f1-score   support

        book       1.00      1.00      1.00         2
       movie       1.00      1.00      1.00         5
  restaurant       1.00      1.00      1.00         3

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10

TF-IDF + LinearSVC profiles
              precision    recall  f1-score   support

        book       1.00      1.00      1.00         2
       movie       1.00      1.00      1.00         5
  restaurant       1.00      1.00      1.00         3

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10



,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1
0,TF-IDF cosine profiles,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,TF-IDF + LinearSVC profiles,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [7]:
confusion_df = pd.DataFrame(
    confusion_matrix(test_df["topic"], test_df["tfidf_cosine_topic"], labels=topic_labels),
    index=[f"gold_{label}" for label in topic_labels],
    columns=[f"pred_{label}" for label in topic_labels],
)
display(confusion_df)

,pred_book,pred_movie,pred_restaurant
gold_book,2,0,0
gold_movie,0,5,0
gold_restaurant,0,0,3


## Qualitative Analysis: Did the Model Use the Right Reasons?

The scores are expected to be very high on this small test set because most sentences contain direct topic cues. The next cell lists the highest-contributing TF-IDF terms for the predicted topic. This helps check whether the model is matching sensible evidence.

In [8]:
feature_names = np.array(tfidf.get_feature_names_out())


def top_similarity_terms(row_index, topic_label, top_n=5):
    topic_index = topic_labels.index(topic_label)
    contribution = test_matrix[row_index].multiply(profile_matrix[topic_index]).toarray().ravel()
    nonzero = contribution.nonzero()[0]
    if len(nonzero) == 0:
        return []
    ordered = nonzero[np.argsort(contribution[nonzero])[::-1]][:top_n]
    return feature_names[ordered].tolist()


evidence_rows = []
for row_index, row in test_df.iterrows():
    predicted_topic = row["tfidf_cosine_topic"]
    evidence_rows.append({
        "sentence_id": row["sentence id"],
        "gold_topic": row["topic"],
        "predicted_topic": predicted_topic,
        "score": row["tfidf_cosine_score"],
        "evidence_terms": ", ".join(top_similarity_terms(row_index, predicted_topic)),
        "text": row["text"],
    })

evidence_df = pd.DataFrame(evidence_rows)
display(evidence_df)

,sentence_id,gold_topic,predicted_topic,score,evidence_terms,text
0,0,movie,movie,0.148554,movie,It took eight years for Warner Brothers to rec...
1,1,restaurant,restaurant,0.291730,"students, soho, diner, atmosphere",All the New York University students love this...
2,2,restaurant,restaurant,0.326164,"trendy, restaurant, place, italian, food",This Italian place is really trendy but they h...
3,3,book,book,0.325081,"review, jane austen, jane, book, austen","In conclusion, my review of this book would be..."
4,4,movie,movie,0.281015,"played, navy, movie, story",The story of this movie is focused on Carl Bra...
5,5,movie,movie,0.257304,"movie, filming, commercial",Chris O'Donnell stated that while filming for ...
6,6,restaurant,restaurant,0.291730,"place, favorite, eat, amsterdam",My husband and I moved to Amsterdam 6 years ag...
7,7,movie,movie,0.257304,"role, performed, movies",Dame Maggie Smith performed her role excellent...
8,8,movie,movie,0.202484,"shot, movie, story",The new movie by Mr. Kruno was shot in New Yor...
9,9,book,book,0.145381,novels,"I always have loved English novels, but I just..."


## Stress Test and Limitations

The project asks for error analysis even if the main result is perfect. Since the gold test set is very small and explicit, the following artificial stress-test sentences show when this TF-IDF approach becomes fragile. These are not part of the graded test set; they are used only for qualitative analysis.

In [9]:
stress_examples = pd.DataFrame({
    "text": [
        "The story was unforgettable and the characters stayed with me for days.",
        "The service was slow, but the atmosphere made the evening enjoyable.",
        "The cast was excellent, although the script needed more work.",
        "I liked the plot, but the ending felt rushed.",
    ],
    "expected_topic": ["book_or_movie", "restaurant", "movie", "book_or_movie"],
})

stress_matrix = tfidf.transform(stress_examples["text"])
stress_similarities = cosine_similarity(stress_matrix, profile_matrix)
stress_examples["predicted_topic"] = [topic_labels[index] for index in stress_similarities.argmax(axis=1)]
stress_examples["score"] = stress_similarities.max(axis=1)
display(stress_examples)

,text,expected_topic,predicted_topic,score
0,The story was unforgettable and the characters...,book_or_movie,movie,0.159777
1,"The service was slow, but the atmosphere made ...",restaurant,restaurant,0.206284
2,"The cast was excellent, although the script ne...",movie,book,0.000000
3,"I liked the plot, but the ending felt rushed.",book_or_movie,book,0.205599


The stress test illustrates the main limitation: TF-IDF does not understand context. It matches words. Sentences containing explicit words like `movie`, `restaurant`, `book`, `food`, `diner`, `actor`, or `novel` are easy. Sentences using more general words like `story`, `plot`, `characters`, or `ending` can be ambiguous because those words are common in both book and movie discussions.

## Save Predictions for the Project Zip

In [10]:
OUTPUT_PATH.parent.mkdir(exist_ok=True)
test_df.to_csv(OUTPUT_PATH, index=False)
summary_df.to_csv(SUMMARY_PATH, index=False)

print(f"Saved sentence-level topic predictions to {OUTPUT_PATH}")
print(f"Saved model summary to {SUMMARY_PATH}")

Saved sentence-level topic predictions to outputs/topic_tfidf_predictions.csv
Saved model summary to outputs/topic_tfidf_summary.csv


## Poster Discussion Notes

**Approach.** Text is represented with TF-IDF, using lowercasing, English stop-word removal, unigrams and bigrams, and sublinear term frequency. The main system predicts the topic whose profile vector has the highest cosine similarity to the sentence vector. The second system uses the same TF-IDF representation with a LinearSVC classifier, following the Lab 6 baseline idea.

**Results.** Both systems reach perfect scores on the provided 10-sentence test set. This should be interpreted cautiously because the test set is very small and contains direct lexical cues for the topics.

**Error analysis / right reasons.** The evidence table shows that predictions are usually driven by appropriate terms: `movie`, `movies`, `filming`, `role`, `book`, `novels`, `restaurant`, `food`, `diner`, and `eat`. However, this also reveals the limitation: the systems rely on surface words rather than deeper topic understanding.

**Limitations and improvements.** The main limitation is the lack of a balanced labelled training set for the three project topics. With more time, we should collect labelled sentence-level examples for book, movie, and restaurant reviews, train a supervised TF-IDF classifier on that data, and test it on a larger and less keyword-obvious test set.